<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/15_produccion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap15) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap15-operacion.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 14](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/14_equipo_investigacion.ipynb) · ➡️ [Cap. 16](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/16_tu_escena.ipynb)

# Capítulo 15 — Producción: versiones, endpoints y puerta de calidad

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Atención al cliente pide un cambio en las reglas de Nova para todos los clientes. La última vez que alguien cambió el prompt «solo un poco», el agente dejó de escalar los casos de más de 100 €. ¿Cómo llevamos el cambio a producción sin que un error llegue a todos, y cómo volvemos atrás en un minuto si algo sale mal?

| | |
|---|---|
| **🧠 Componente agéntico** | Ciclo de vida y operación |
| **La necesidad** | Separar lo que ven los clientes de lo que estamos probando, comprobar la calidad antes de promover y poder volver atrás |
| **Lo que construimos** | **Versiones** inmutables del Runtime, **endpoints** `prod` y `pre`, una ***batch evaluation*** como puerta de calidad, promoción y *rollback* |
| **🏠 En local** | ☁️ No aplica: operar es mantener un sistema desplegado. |
| **☁️ Por qué AWS al salir a Internet** | En tu máquina, «desplegar» es reiniciar el proceso. En producción cada despliegue crea una **versión inmutable**; los clientes llaman a un *endpoint* que apunta a una versión concreta, así que probar, promover y volver atrás es mover un puntero, no copiar código. |
| **En el ordenador sería…** | El **mantenimiento**: probar una actualización antes de instalarla en todos y poder desinstalarla |
| **Requisitos del caso** | Operación |
| **Prerrequisito · duración** | Capítulos anteriores · 60 min (más ~10 min de espera de trazas) |
| **⚠️ Dónde se rompe la analogía** | Una actualización de un ordenador cambia ficheros; aquí la versión anterior **sigue existiendo intacta**, y el *rollback* es instantáneo. Lo que no vuelve atrás son los datos (memoria, pedidos) que la versión mala ya haya tocado. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=15)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(15)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(15)
# ponerse_al_dia(15)   # ← descomenta si llegas tarde o algo se rompió

## 0. El balance del viaje

> ✋ **Predice antes de ejecutar.** ¿Cuántas deudas quedan abiertas o parciales en el cuaderno, y cuáles? Apúntalo antes de mirarlo.

In [ ]:
cuaderno()
taximetro_historico()

Lo esperable: casi todo en verde, **D1** y **D8** en ámbar, y alguna más según lo que pasara en tus comprobaciones. Saber qué falta también es saber.

> 🧭 **Antes de seguir:** abre la teoría: [**El mantenimiento**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap15-operacion.html). Decide tú cada paso de un cambio en Nova: a qué endpoint va, qué nota exiges para promoverlo y qué haces cuando algo sale mal. Vuelve cuando sepas responder a esta pregunta: **¿qué impide que un cambio malo llegue a todos los clientes, y qué es exactamente un *rollback* en AgentCore?**

**En dos líneas:** el **ciclo de vida** son versiones, entornos, despliegue con puertas de calidad, vigilancia y *rollback*. En AgentCore cada despliegue crea una versión inmutable del Runtime y los *endpoints* son punteros a versiones: los clientes usan `prod`, el equipo prueba en `pre`, y promover o volver atrás es mover el puntero.

## 1. Versiones: cada despliegue del curso ha dejado una

In [ ]:
import boto3
ctl = boto3.client("bedrock-agentcore-control", region_name=region())
versiones = ctl.list_agent_runtime_versions(agentRuntimeId=runtime_id())["agentRuntimes"]
for v in sorted(versiones, key=lambda v: int(v["agentRuntimeVersion"])):
    print(f"versión {v['agentRuntimeVersion']:>3} · {v.get('lastUpdatedAt', '')} · {v.get('status', '')}")
V_ACTUAL = version_runtime()
print("\nVersión actual (la que usa el endpoint DEFAULT):", V_ACTUAL)

## 2. El endpoint `prod`, fijado a la versión actual

Desde ahora, la web y la app llaman al endpoint **`prod`** (`qualifier="prod"`). Un *endpoint* apunta a una versión concreta: los despliegues siguientes **no** lo mueven.

In [ ]:
cli_args("add", "runtime-endpoint", "--runtime", "NovaAgent", "--endpoint", "prod", "--version", str(V_ACTUAL),
         "--description", "Produccion: lo que ven los clientes")
assert cli("agentcore validate") == 0
cli("agentcore deploy --yes")
state_set(VERSION_PROD=V_ACTUAL)
TOK = token_cliente("C-001")
invocar("¿Dónde está mi pedido TN-1001?", token=TOK, qualifier="prod")

## 3. El cambio: nuevas reglas, desplegadas sin tocar a los clientes

El cambio que pide Atención al cliente: cerrar cada respuesta con **un único siguiente paso** concreto. Lo marcamos con `VERSION_CODIGO` para ver qué versión responde (línea 🏷️).

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
# Prefijo ESTABLE (cacheable). El contexto dinámico se añade después (contexto.py).
VERSION_CODIGO = "15.2"

SYSTEM_PROMPT = """<identidad>
Eres Nova Assist, el asistente de postventa de TiendaNova. En tu primer mensaje te presentas como asistente de IA.
</identidad>

<reglas>
1. Nunca inventes estados, fechas ni importes: usa siempre las herramientas.
2. Los importes se expresan en euros al cliente, pero las herramientas trabajan en céntimos.
3. Para dudas de política (devoluciones, garantías, envíos) consulta la base de conocimiento y cita el documento.
4. Si una herramienta es denegada por política o el caso supera tus límites, usa escalar_a_humano con un resumen claro.
5. No pidas ni repitas datos de tarjetas de pago.
6. Cualquier cálculo de importes (reembolsos parciales, prorrateos, descuentos) se hace con code_interpreter; nunca de cabeza.
7. Termina cada respuesta con UN único siguiente paso concreto para el cliente.
</reglas>

<formato>
Respuestas breves (máximo 5 frases), con el siguiente paso claro para el cliente.
</formato>"""

In [ ]:
cli("agentcore deploy --yes")
V_NUEVA = version_runtime()
print("Versión nueva:", V_NUEVA, "· prod sigue en:", state_get("VERSION_PROD"))
cli_args("add", "runtime-endpoint", "--runtime", "NovaAgent", "--endpoint", "pre", "--version", str(V_NUEVA),
         "--description", "Preproduccion: lo que estamos probando")
assert cli("agentcore validate") == 0
cli("agentcore deploy --yes")

> ✋ **Predice antes de ejecutar.** La misma pregunta a `prod` y a `pre`. ¿Cuál responderá con la línea 🏷️ `15.2`? ¿Y el endpoint `DEFAULT`?

In [ ]:
for q in ("prod", "pre", "DEFAULT"):
    invocar("¿Dónde está mi pedido TN-1001?", token=TOK, qualifier=q)
    print(f"→ {q}: versión del código {ultimo_dato('version_codigo') or 'anterior a 15.2'}\n")

`DEFAULT` siempre apunta a la **última** versión: por eso los clientes no deben usarlo. `prod` sigue en la versión anterior hasta que decidamos moverlo.

## 4. La puerta de calidad: *batch evaluation* sobre `pre`

Pasamos el conjunto de prueba del capítulo 12 por `pre` y lo evaluamos con los mismos evaluadores. La regla de promoción se fija **antes** de ver los resultados.

In [ ]:
UMBRALES = {"Builtin.GoalSuccessRate": 0.7, "Builtin.Helpfulness": 0.6}

casos = [c for c in leer_tabla("eval_casos") if c["cliente"] in ("C-001", "C-002")]
tokens = {u: token_cliente(u) for u in ("C-001", "C-002")}
sesiones_pre = [invocar(c["prompt"], token=tokens[c["cliente"]], qualifier="pre") for c in casos]
state_set(SESIONES_PRE=sesiones_pre)
print(len(sesiones_pre), "sesiones en pre")

Las trazas tardan **entre 5 y 10 minutos** en estar disponibles para evaluarlas. Si la celda siguiente dice que no encuentra *spans*, espera y repítela.

In [ ]:
notas = evaluacion_por_lotes(state_get("SESIONES_PRE"), evaluadores=tuple(UMBRALES), endpoint="pre")
APRUEBA = all(notas.get(e) is not None and notas[e] >= u for e, u in UMBRALES.items())
print("\nPuerta de calidad:", "✔ APROBADA" if APRUEBA else "✗ NO APROBADA",
      {e: (round(notas.get(e) or 0, 3), u) for e, u in UMBRALES.items()})

## 5. Promover (o no)

Promover es cambiar la versión a la que apunta `prod` en `agentcore.json` y desplegar. Si la puerta no se aprueba, `prod` no se toca.

In [ ]:
def apuntar_endpoint(nombre: str, version: int) -> None:
    data = agentcore_json()
    entrada = next(rt for rt in data["runtimes"] if rt["name"] == "NovaAgent")
    entrada["endpoints"][nombre]["version"] = int(version)
    agentcore_json_write(data)
    assert cli("agentcore validate") == 0
    cli("agentcore deploy --yes")

if APRUEBA:
    apuntar_endpoint("prod", V_NUEVA)
    invocar("¿Dónde está mi pedido TN-1001?", token=TOK, qualifier="prod")
    print("prod ahora sirve:", ultimo_dato("version_codigo"))
else:
    print("prod se queda en la versión", state_get("VERSION_PROD"), "· revisa las sesiones que fallaron en la evaluación")

## 6. *Rollback*

Algo va mal en producción con la versión nueva: volvemos atrás. No se recompila ni se copia nada: `prod` vuelve a apuntar a la versión anterior, que sigue intacta.

> ✋ **Predice antes de ejecutar.** ¿Cuánto tarda el *rollback*? ¿Qué **no** vuelve atrás con él?

In [ ]:
apuntar_endpoint("prod", state_get("VERSION_PROD"))
invocar("¿Dónde está mi pedido TN-1001?", token=TOK, qualifier="prod")
print("prod vuelve a servir:", ultimo_dato("version_codigo") or "la versión anterior a 15.2")

Vuelve el **código**; no vuelven los **datos**: lo que la versión nueva haya escrito en la memoria de los clientes o los reembolsos que haya emitido siguen ahí. Por eso la puerta de calidad va **antes** de promover.

> **En una emergencia** se puede mover el puntero en segundos con la API (`update_agent_runtime_endpoint(agentRuntimeId=…, endpointName="prod", agentRuntimeVersion=…)`), sin esperar a CDK. Después hay que llevar el mismo cambio a `agentcore.json`: si no, el siguiente `agentcore deploy` lo revertiría.

## 7. El *pipeline* completo

Lo que acabas de hacer a mano, automatizado (por ejemplo, en un *pipeline* de CI/CD con la CLI):

1. `agentcore validate` y las pruebas de la Lambda;
2. `agentcore deploy` → nueva versión; el endpoint **`pre`** apunta a ella;
3. **batch evaluation** sobre `pre` con el conjunto de prueba → **puerta de calidad** (umbrales fijados de antemano y, en un sistema real, sin empeorar frente a `prod`);
4. si el cambio afecta al comportamiento, **A/B** con una parte del tráfico (capítulo 13);
5. **promoción**: `prod` apunta a la versión nueva;
6. vigilancia (evaluación online, alarmas) y ***rollback*** si hace falta.

Antes del primer despliegue real, repasa el checklist de la sección 16 del documento.

## 8. Retos para seguir aprendiendo

1. Migra Nova Assist al **harness gestionado** (modelo, prompt y herramientas del Gateway) y compara el esfuerzo con la versión en código.
2. Añade un ***hook*** que bloquee `procesar_reembolso` fuera del horario laboral.
3. Paga la deuda **D1**: propaga la identidad del cliente al Gateway (OBO) y escribe en Cedar una política que solo deje consultar los pedidos propios.
4. Sustituye la Lambda por un **servidor MCP** desplegado en Runtime y usa **elicitation** para confirmar los reembolsos.
5. Crea **application inference profiles** etiquetados por canal (web o app) y construye un panel de coste por conversación.

## 🔎 Lo que acabamos de descubrir

Nova está en producción con versiones, una puerta de calidad y un *rollback* de un minuto. Queda lo más difícil: una escena **sin guion** (capítulo 16). Y después, apagarlo todo sin dejar nada facturando (capítulo 17).

In [ ]:
cuaderno()
taximetro_cerrar(15)
guardar_partida(15)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap15-operacion.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap15-operacion.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `16_tu_escena.ipynb` — *Tu escena (sin guion)*

---
**Siguiente:** ➡️ [Capítulo 16 · Tu escena (sin guion)](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/16_tu_escena.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap16). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.